#### Setup

In [ ]:
%pip install -q transformers accelerate datasets einops jaxtyping

In [ ]:
%pip uninstall -y torchvision

In [1]:
import os
import sys
from pathlib import Path

import einops
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from jaxtyping import Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
# Colab CPU setup
root = Path("/content")
workdir = root / "linear_probes"

workdir.mkdir(exist_ok=True)
os.chdir(workdir)

# Clone required repos
if not Path("geometry-of-truth").exists():
    !git clone https://github.com/saprmarks/geometry-of-truth.git

if not Path("deception-detection").exists():
    !git clone https://github.com/ApolloResearch/deception-detection.git

GOT_ROOT = workdir / "geometry-of-truth"
DD_ROOT = workdir / "deception-detection"

GOT_DATASETS = GOT_ROOT / "datasets"
DD_DATA = DD_ROOT / "data"

assert GOT_ROOT.exists()
assert DD_ROOT.exists()
assert GOT_DATASETS.exists()

In [3]:
print("GOT datasets:", GOT_DATASETS)
print("DD data:", DD_DATA)

GOT datasets: /content/linear_probes/geometry-of-truth/datasets
DD data: /content/linear_probes/deception-detection/data


#### Model loading

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

device = t.device("cpu")
dtype = t.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model.to(device)
model.eval()

NUM_LAYERS = len(model.model.layers)
D_MODEL = model.config.hidden_size

PROBE_LAYER = NUM_LAYERS // 2
INTERVENE_LAYER = max(0, PROBE_LAYER - 4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [5]:
print(f"Probe layer: {PROBE_LAYER}\nIntervene layer: {INTERVENE_LAYER}")

Probe layer: 12
Intervene layer: 8


In [6]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(GOT_DATASETS / f"{name}.csv")
    datasets[name] = df
    print(f"\n{name}: {len(df)} statements ({df['label'].sum()} true, {(1 - df['label']).sum():.0f} false)")
    display(df.head(4))


cities: 1496 statements (748 true, 748 false)


,statement,label,city,country,correct_country
0,The city of Krasnodar is in Russia.,1,Krasnodar,Russia,Russia
1,The city of Krasnodar is in South Africa.,0,Krasnodar,South Africa,Russia
2,The city of Lodz is in Poland.,1,Lodz,Poland,Poland
3,The city of Lodz is in the Dominican Republic.,0,Lodz,the Dominican Republic,Poland



sp_en_trans: 354 statements (177 true, 177 false)


,statement,label
0,The Spanish word 'con' means 'to speak'.,0
1,The Spanish word 'uno' means 'one'.,1
2,The Spanish word 'tener' means 'to have'.,1
3,The Spanish word 'caliente' means 'hot'.,1



larger_than: 1980 statements (990 true, 990 false)


,statement,label,n1,n2,diff,abs_diff
0,Fifty-one is larger than fifty-two.,0,51,52,-1,1
1,Fifty-one is larger than fifty-three.,0,51,53,-2,2
2,Fifty-one is larger than fifty-four.,0,51,54,-3,3
3,Fifty-one is larger than fifty-five.,0,51,55,-4,4


#### Extracting activations

In [14]:
from tqdm import tqdm

def extract_activations(
    statements: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    layers: list[int],
    batchSize: int = 20,
    desc: str = "Extracting activations"
) -> dict[int, Tensor]:

    """
    output -> {layer_number: activation_tensor}
    """

    activationStore = {layer: [] for layer in layers}

    for i in tqdm(
        range(0, len(statements), batchSize),
        desc=desc,
        leave=False
    ):
        batch = statements[i:i + batchSize]

        for statement in batch:
            assert statement.rstrip().endswith("."), (
                f"Statement does not end with period: {statement}"
            )

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        with t.no_grad():
            outputs = model(
                **inputs,
                output_hidden_states=True
            )

        last_real_token_indices = (
            inputs["attention_mask"].sum(dim=1) - 1
        )

        for layer in layers:
            hiddenStates = outputs.hidden_states[layer + 1]

            batchIndices = t.arange(
                hiddenStates.shape[0],
                device=hiddenStates.device
            )

            activations = hiddenStates[
                batchIndices,
                last_real_token_indices
            ]

            activationStore[layer].append(
                activations.cpu().float()
            )

    return {
        layer: t.cat(acts_list, dim=0)
        for layer, acts_list in activationStore.items()
    }

In [15]:
activations = {}
labels_dict = {}
statements_dict = {}

for name in DATASET_NAMES:
    print(f"\nProcessing dataset: {name}")

    df = datasets[name]
    statements = df["statement"].tolist()
    labs = t.tensor(df["label"].values, dtype=t.float32)

    statements_dict[name] = statements

    acts = extract_activations(
        statements,
        model,
        tokenizer,
        [PROBE_LAYER]
    )

    activations[name] = acts[PROBE_LAYER]
    labels_dict[name] = labs


Processing dataset: cities



Processing dataset: sp_en_trans



Processing dataset: larger_than
